# 缩放点积注意力

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Scaled Dot-Product Attention
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: [batch_size, n_heads, seq_len_q, d_k]
            key: [batch_size, n_heads, seq_len_k, d_k]
            value: [batch_size, n_heads, seq_len_v, d_v]
            mask: [batch_size, 1, seq_len_q, seq_len_k] (optional)
        
        Returns:
            output: [batch_size, n_heads, seq_len_q, d_v]
            attention_weights: [batch_size, n_heads, seq_len_q, seq_len_k]
        """
        # Compute scaled dot-product attention scores
        d_k = query.size(-1)  # Dimension of key/query

        # calculate the dot product attention scores, where transpose(-2, -1) is used to swap the last two dimensions of the key tensor.
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
        
        # Apply mask (if provided)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))  # Mask out invalid positions
        
        # Softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Weighted sum of values
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights



import torch

def main():
    # 设置随机种子以确保结果可重复
    torch.manual_seed(42)

    # 定义超参数
    batch_size = 2
    n_heads = 4
    seq_len_q = 5
    seq_len_k = 5
    seq_len_v = 5
    d_k = 8  # Dimension of query/key
    d_v = 16  # Dimension of value

    # 创建随机输入张量
    query = torch.randn(batch_size, n_heads, seq_len_q, d_k)
    key = torch.randn(batch_size, n_heads, seq_len_k, d_k)
    value = torch.randn(batch_size, n_heads, seq_len_v, d_v)

    # 可选：创建一个掩码 (mask)
    mask = torch.ones(batch_size, 1, seq_len_q, seq_len_k)  # 全1表示没有遮挡
    mask[:, :, 2:, :] = 0  # 遮挡部分位置

    # 初始化 Scaled Dot-Product Attention 模块
    attention_module = ScaledDotProductAttention(dropout=0.1)

    # 前向传播
    output, attention_weights = attention_module(query, key, value, mask)

    # 打印输出形状和注意力权重
    print("Output shape:", output.shape)  # 应为 [batch_size, n_heads, seq_len_q, d_v]
    print("Attention weights shape:", attention_weights.shape)  # 应为 [batch_size, n_heads, seq_len_q, seq_len_k]

    # 打印部分输出和注意力权重
    print("\nOutput (first head):")
    print(output[0, 0])  # 第一个样本的第一个头
    print("\nAttention weights (first head):")
    print(attention_weights[0, 0])  # 第一个样本的第一个头


if __name__ == "__main__":
    main()

Output shape: torch.Size([2, 4, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])

Output (first head):
tensor([[ 0.3545, -0.4134,  0.7275,  0.1496, -0.0250,  0.2935, -0.3197,  0.3172,
         -1.0895, -0.2941, -0.0215, -0.3014, -0.1695,  0.8140,  0.2216,  0.1581],
        [ 0.1937,  0.0703,  1.0720,  0.5125, -0.0382,  1.0846, -0.4019, -0.5019,
         -1.2636, -0.0994, -0.3917, -0.5174, -0.4077,  0.7188,  0.7184,  0.5155],
        [    nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan],
        [    nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan],
        [    nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan,
             nan,     nan,     nan,     nan,     nan,     nan,     nan,     nan]])

Attention weights (first head):
tensor([[0.2847, 0.1589, 0.3397, 0.2617

# 多头注意力机制

In [6]:

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model  # Embedding dimension
        self.n_heads = n_heads  # Number of heads
        self.d_k = d_model // n_heads  # Dimension of each head
        
        # Linear layers for Query, Key, Value, and final output
        self.linear_q = nn.Linear(d_model, d_model)
        self.linear_k = nn.Linear(d_model, d_model)
        self.linear_v = nn.Linear(d_model, d_model)
        self.linear_out = nn.Linear(d_model, d_model)
        
        # Attention mechanism
        self.attention = ScaledDotProductAttention(dropout)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """
        Split the last dimension into (n_heads, d_k).
        Transpose to shape [batch_size, n_heads, seq_len, d_k].
        """
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # [batch_size, n_heads, seq_len, d_k]

    def combine_heads(self, x):
        """
        Combine the heads back to original shape.
        Input shape: [batch_size, n_heads, seq_len, d_k]
        Output shape: [batch_size, seq_len, d_model]
        """
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()  # [batch_size, seq_len, n_heads, d_k]
        return x.view(batch_size, seq_len, self.d_model)  # [batch_size, seq_len, d_model]

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: [batch_size, seq_len_q, d_model]
            key: [batch_size, seq_len_k, d_model]
            value: [batch_size, seq_len_v, d_model]
            mask: [batch_size, 1, seq_len_q, seq_len_k] (optional)
        
        Returns:
            output: [batch_size, seq_len_q, d_model]
            attention_weights: [batch_size, n_heads, seq_len_q, seq_len_k]
        """
        # Linear projections
        query = self.linear_q(query)
        key = self.linear_k(key)
        value = self.linear_v(value)
        
        # Split heads
        query = self.split_heads(query)  # [batch_size, n_heads, seq_len_q, d_k]
        key = self.split_heads(key)      # [batch_size, n_heads, seq_len_k, d_k]
        value = self.split_heads(value)  # [batch_size, n_heads, seq_len_v, d_v]
        
        # Apply attention
        attention_output, attention_weights = self.attention(query, key, value, mask)
        
        # Combine heads
        attention_output = self.combine_heads(attention_output)  # [batch_size, seq_len_q, d_model]
        
        # Final linear layer
        output = self.linear_out(attention_output)
        output = self.dropout(output)
        
        return output, attention_weights
    


# Example Usage
if __name__ == "__main__":
    # Hyperparameters
    batch_size = 2
    seq_len = 5
    d_model = 8
    n_heads = 2
    
    # Random input tensors
    query = torch.rand(batch_size, seq_len, d_model)
    key = torch.rand(batch_size, seq_len, d_model)
    value = torch.rand(batch_size, seq_len, d_model)
    
    # Initialize Multi-Head Attention
    mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
    
    # Forward pass
    output, attention_weights = mha(query, key, value)
    
    print("Output Shape:", output.shape)  # Expected: [batch_size, seq_len, d_model]
    print("Attention Weights Shape:", attention_weights.shape)  # Expected: [batch_size, n_heads, seq_len, seq_len]


Output Shape: torch.Size([2, 5, 8])
Attention Weights Shape: torch.Size([2, 2, 5, 5])
